# 03 — Hero & Composition Analysis: Does Map Geometry Dictate Comp Viability?

One of the core strategic questions in competitive Overwatch:

> **"Map geometry dictates composition viability; hero synergies matter more than individual hero strength."**
> — Widely accepted coaching principle

### Key OW Concepts
- **Composition (comp)**: The team's 5-hero lineup. In OW2: 1 Tank, 2 DPS, 2 Support.
- **Archetypes**: Dive (fast engagement), Brawl (close-range deathball), Poke (long-range spam), Mixed.
- **Synergy**: Certain hero pairings amplify each other (e.g., Winston + Tracer for dive, Reinhardt + Lucio for brawl).
- **Hero swap**: Unlike other FPS games, OW allows mid-match hero changes. Knowing when to swap is a key skill.
- **Map geometry**: Sightlines, verticality, choke points, and flank routes determine which heroes thrive.

### Analysis Plan
1. Hero pick rates and win rates
2. Composition archetype tagging and win rates
3. Hero synergy matrix (pair co-occurrence x win rate)
4. Map-specific hero preferences
5. Hero swap patterns and their outcomes
6. Coaching implications

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from src.data_loader import load_csv, load_matches
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES,
    classify_composition
)
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig, role_color

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load Data

We need several tables:
- **HeroSpawn**: Every time a player spawns as a hero (initial spawn + respawns)
- **HeroSwap**: When a player switches heroes mid-match
- **MatchStart / MatchEnd**: For map info, scores, and determining winners
- **PlayerStat**: Per-player per-hero stats including time played

In [ ]:
hero_spawn = load_csv('HeroSpawn')
hero_swap = load_csv('HeroSwap')
player_stats = load_csv('PlayerStat')
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)

print(f"HeroSpawn events: {len(hero_spawn):,}")
print(f"HeroSwap events:  {len(hero_swap):,}")
print(f"PlayerStat rows:  {len(player_stats):,}")
print(f"Matches:          {len(matches):,}")
print()
print("Maps in dataset:", matches['map_name'].nunique())
print("Map types:", matches['map_type'].unique().tolist())

## 2. Hero Pick Rates and Win Rates

We use PlayerStat to find per-hero time played across all matches, then join with match outcomes to compute win rates. A hero's "pick rate" is the fraction of matches where they were played for a meaningful amount of time.

In [ ]:
# Join player stats with match outcomes
ps = player_stats.merge(
    matches[['MapDataId', 'winner', 'team_1_name', 'team_2_name', 'map_name', 'map_type']],
    on='MapDataId',
    how='inner'
)

# Filter to meaningful playtime (at least 60 seconds on a hero)
ps = ps[ps['hero_time_played'] >= 60].copy()

# Determine if the player's team won
ps['team_won'] = ps['player_team'] == ps['winner']
ps = add_role_column(ps)

# Hero pick rates: fraction of matches where hero was played
total_matches = matches['MapDataId'].nunique()
hero_picks = ps.groupby('player_hero').agg(
    matches_played=('MapDataId', 'nunique'),
    total_time=('hero_time_played', 'sum'),
    wins=('team_won', 'sum'),
    total=('team_won', 'count'),
    role=('role', 'first')
)
hero_picks['pick_rate'] = hero_picks['matches_played'] / total_matches * 100
hero_picks['win_rate'] = hero_picks['wins'] / hero_picks['total'] * 100
hero_picks = hero_picks.sort_values('pick_rate', ascending=False)

print(f"Unique heroes: {len(hero_picks)}")
print(f"Total matches: {total_matches}")
print()
print("Top 15 most picked heroes:")
print(hero_picks.head(15)[['pick_rate', 'win_rate', 'total', 'role']].to_string())

In [ ]:
# Visualization: Hero pick rates with win rates
top20 = hero_picks.head(20).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Pick rate
colors = [role_color(r) for r in top20['role']]
axes[0].barh(top20.index[::-1], top20['pick_rate'].values[::-1],
             color=colors[::-1], alpha=0.85)
axes[0].set_xlabel('Pick Rate (%)')
axes[0].set_title('Top 20 Heroes by Pick Rate')

# Win rate (all heroes with enough data)
wr_data = hero_picks[hero_picks['total'] >= 30].sort_values('win_rate', ascending=True)
colors_wr = [role_color(r) for r in wr_data['role']]
axes[1].barh(wr_data.index, wr_data['win_rate'], color=colors_wr, alpha=0.85)
axes[1].axvline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='50%')
axes[1].set_xlabel('Win Rate (%)')
axes[1].set_title('Hero Win Rates (min 30 games)')
axes[1].legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ['Tank', 'DPS', 'Support']]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '03_hero_pick_win_rates')
plt.show()

## 3. Composition Archetype Win Rates

Using `classify_composition()`, we tag each team's lineup per match into an archetype (Dive, Brawl, Poke, or Mixed). We determine the lineup from the heroes with the most playtime for each player in a match.

In [ ]:
# For each match+team, find the primary hero for each player (most time played)
idx = ps.groupby(
    ['MapDataId', 'player_team', 'player_name'], observed=True
)['hero_time_played'].idxmax()
primary_heroes = ps.loc[idx].reset_index(drop=True)

# Get team compositions per match
team_comps = primary_heroes.groupby(
    ['MapDataId', 'player_team'], observed=True
).agg(
    heroes=('player_hero', list),
    team_won=('team_won', 'first')
).reset_index()

# Classify compositions
team_comps['archetype'] = team_comps['heroes'].apply(classify_composition)
team_comps['hero_count'] = team_comps['heroes'].apply(len)

print("Composition archetype distribution:")
print(team_comps['archetype'].value_counts())
print(f"\nTotal team-match observations: {len(team_comps):,}")

In [ ]:
# Win rate by archetype
arch_wr = team_comps.groupby('archetype').agg(
    total=('team_won', 'count'),
    wins=('team_won', 'sum')
)
arch_wr['win_rate'] = arch_wr['wins'] / arch_wr['total'] * 100
arch_wr = arch_wr.sort_values('win_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
arch_colors = {
    'Dive': OW_COLORS['teal'],
    'Brawl': OW_COLORS['red'],
    'Poke': OW_COLORS['blue'],
    'Mixed': OW_COLORS['light_gray'],
}
colors = [arch_colors.get(a, OW_COLORS['orange']) for a in arch_wr.index]

bars = ax.barh(arch_wr.index, arch_wr['win_rate'], color=colors, alpha=0.85, height=0.5)
ax.axvline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='50%')

for bar, (arch, row) in zip(bars, arch_wr.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row["win_rate"]:.1f}% (n={row["total"]:,})',
            va='center', fontsize=11, color=OW_COLORS['white'])

ax.set_xlabel('Win Rate (%)')
ax.set_title('Composition Archetype Win Rates')
ax.legend()

plt.tight_layout()
save_fig(fig, '03_comp_archetype_win_rates')
plt.show()

## 4. Hero Synergy Matrix

Which hero pairs work best together? We build a co-occurrence matrix for the top 20 most-played heroes, then compute the win rate when those two heroes appear on the same team in the same match.

**How to read the heatmap**: Each cell shows the win rate when both heroes are on the same team. Values above 50% indicate positive synergy; below 50% suggests the pairing underperforms.

In [ ]:
# Get top 20 heroes by pick rate for the synergy matrix
top_heroes = hero_picks.head(20).index.tolist()

# Filter primary heroes to top 20 only (big speed improvement)
top_primary = primary_heroes[primary_heroes['player_hero'].isin(top_heroes)].copy()

# Build pair records efficiently
team_heroes = top_primary.groupby(
    ['MapDataId', 'player_team'], observed=True
).agg(
    heroes=('player_hero', list),
    team_won=('team_won', 'first')
).reset_index()

# Generate pairs from the hero lists (skip teams with < 2 top heroes)
pair_records = []
for _, row in team_heroes.iterrows():
    heroes = row['heroes']
    if not isinstance(heroes, list) or len(heroes) < 2:
        continue
    heroes = sorted(set(heroes))
    won = row['team_won']
    for h1, h2 in combinations(heroes, 2):
        pair_records.append({'hero_1': h1, 'hero_2': h2, 'won': won})

pairs_df = pd.DataFrame(pair_records)
print(f"Total hero pair observations: {len(pairs_df):,}")

# Compute pair win rates
pair_wr = pairs_df.groupby(['hero_1', 'hero_2']).agg(
    total=('won', 'count'),
    wins=('won', 'sum')
)
pair_wr['win_rate'] = pair_wr['wins'] / pair_wr['total'] * 100
pair_wr = pair_wr[pair_wr['total'] >= 10]  # Minimum sample size
pair_wr = pair_wr.reset_index()

print(f"Hero pairs with sufficient data: {len(pair_wr):,}")

In [ ]:
# Build the synergy matrix
synergy_matrix = pd.DataFrame(index=top_heroes, columns=top_heroes, dtype=float)

for _, row in pair_wr.iterrows():
    if row['hero_1'] in top_heroes and row['hero_2'] in top_heroes:
        synergy_matrix.loc[row['hero_1'], row['hero_2']] = row['win_rate']
        synergy_matrix.loc[row['hero_2'], row['hero_1']] = row['win_rate']

# Fill diagonal with individual hero win rates
for hero in top_heroes:
    if hero in hero_picks.index:
        synergy_matrix.loc[hero, hero] = hero_picks.loc[hero, 'win_rate']

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(synergy_matrix, dtype=bool), k=1)

sns.heatmap(
    synergy_matrix.astype(float),
    mask=mask,
    annot=True, fmt='.0f', cmap='RdYlGn', center=50,
    vmin=30, vmax=70,
    linewidths=0.5, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Win Rate (%)'}
)
ax.set_title('Hero Synergy Matrix: Pair Win Rates (Top 20 Heroes)', fontsize=14)

plt.tight_layout()
save_fig(fig, '03_hero_synergy_heatmap')
plt.show()

In [ ]:
# Top and bottom synergy pairs
print("=" * 60)
print("TOP 15 HERO SYNERGY PAIRS (highest win rate together)")
print("=" * 60)
top_pairs = pair_wr.nlargest(15, 'win_rate')
for _, row in top_pairs.iterrows():
    print(f"  {row['hero_1']:>15s} + {row['hero_2']:<15s}  "
          f"Win Rate: {row['win_rate']:.1f}%  (n={row['total']})")

print()
print("=" * 60)
print("BOTTOM 15 HERO SYNERGY PAIRS (lowest win rate together)")
print("=" * 60)
bottom_pairs = pair_wr.nsmallest(15, 'win_rate')
for _, row in bottom_pairs.iterrows():
    print(f"  {row['hero_1']:>15s} + {row['hero_2']:<15s}  "
          f"Win Rate: {row['win_rate']:.1f}%  (n={row['total']})")

## 5. Map-Specific Hero Preferences

Certain heroes thrive on certain maps due to sightlines, verticality, and objective types. We look at which heroes have significantly different pick rates or win rates across maps.

In [ ]:
# Hero pick rate by map (using primary heroes)
map_hero = primary_heroes.groupby(['map_name', 'player_hero']).agg(
    count=('MapDataId', 'count'),
    wins=('team_won', 'sum')
).reset_index()
map_hero['win_rate'] = map_hero['wins'] / map_hero['count'] * 100

# Total games per map for normalization
map_totals = primary_heroes.groupby('map_name')['MapDataId'].nunique().reset_index()
map_totals.columns = ['map_name', 'total_games']
map_hero = map_hero.merge(map_totals, on='map_name')
map_hero['pick_rate'] = map_hero['count'] / map_hero['total_games'] * 100

# Focus on top 10 heroes and maps with enough data
top10_heroes = hero_picks.head(10).index.tolist()
top_maps = map_totals.nlargest(10, 'total_games')['map_name'].tolist()

map_hero_pivot = map_hero[
    map_hero['player_hero'].isin(top10_heroes) &
    map_hero['map_name'].isin(top_maps)
].pivot_table(
    index='map_name', columns='player_hero', values='pick_rate', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    map_hero_pivot,
    annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Pick Rate (%)'}
)
ax.set_title('Hero Pick Rate by Map (Top 10 Heroes, Top 10 Maps)', fontsize=14)
ax.set_ylabel('Map')
ax.set_xlabel('Hero')

plt.tight_layout()
save_fig(fig, '03_map_hero_pick_rates')
plt.show()

In [ ]:
# Same for win rates
map_hero_wr_pivot = map_hero[
    map_hero['player_hero'].isin(top10_heroes) &
    map_hero['map_name'].isin(top_maps) &
    (map_hero['count'] >= 5)  # Minimum sample
].pivot_table(
    index='map_name', columns='player_hero', values='win_rate', fill_value=np.nan
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    map_hero_wr_pivot,
    annot=True, fmt='.0f', cmap='RdYlGn', center=50,
    vmin=30, vmax=70,
    linewidths=0.5, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Win Rate (%)'}
)
ax.set_title('Hero Win Rate by Map (Top 10 Heroes, Top 10 Maps)', fontsize=14)
ax.set_ylabel('Map')
ax.set_xlabel('Hero')

plt.tight_layout()
save_fig(fig, '03_map_hero_win_rates')
plt.show()

## 6. Hero Swap Patterns

Hero swapping is a unique OW mechanic. When do players swap, what do they swap to, and does swapping help or hurt outcomes?

In [ ]:
# Enrich hero swaps with match outcome
swaps = hero_swap.merge(
    matches[['MapDataId', 'winner', 'map_name', 'map_type']],
    on='MapDataId',
    how='inner'
)
swaps['team_won'] = swaps['player_team'] == swaps['winner']
swaps = add_role_column(swaps, hero_col='player_hero')
swaps['prev_role'] = swaps['previous_hero'].map(HERO_ROLES).fillna('Unknown')

print(f"Total hero swaps: {len(swaps):,}")
print(f"Swaps per match (mean): {swaps.groupby('MapDataId').size().mean():.1f}")
print()

# Most common swap patterns
swap_patterns = swaps.groupby(['previous_hero', 'player_hero']).agg(
    count=('id', 'count'),
    wins=('team_won', 'sum')
).reset_index()
swap_patterns['win_rate'] = swap_patterns['wins'] / swap_patterns['count'] * 100
swap_patterns = swap_patterns.sort_values('count', ascending=False)

print("Top 15 most common hero swaps:")
for _, row in swap_patterns.head(15).iterrows():
    print(f"  {row['previous_hero']:>15s} -> {row['player_hero']:<15s}  "
          f"Count: {row['count']:>4d}  Win Rate: {row['win_rate']:.1f}%")

In [ ]:
# Does swapping correlate with winning?
# Compare swap count between winning and losing teams
team_swap_counts = swaps.groupby(['MapDataId', 'player_team']).size().reset_index(name='swaps')
team_swap_counts = team_swap_counts.merge(
    matches[['MapDataId', 'winner']],
    on='MapDataId',
    how='left'
)
team_swap_counts['team_won'] = team_swap_counts['player_team'] == team_swap_counts['winner']

fig, ax = plt.subplots(figsize=(10, 6))
winner_swaps = team_swap_counts[team_swap_counts['team_won']]['swaps']
loser_swaps = team_swap_counts[~team_swap_counts['team_won']]['swaps']

ax.hist(winner_swaps, bins=range(0, 30), alpha=0.6, color=OW_COLORS['green'],
        label=f'Winners (mean: {winner_swaps.mean():.1f})', density=True)
ax.hist(loser_swaps, bins=range(0, 30), alpha=0.6, color=OW_COLORS['red'],
        label=f'Losers (mean: {loser_swaps.mean():.1f})', density=True)
ax.set_xlabel('Hero Swaps per Match')
ax.set_ylabel('Density')
ax.set_title('Hero Swap Frequency: Winners vs Losers')
ax.legend()

plt.tight_layout()
save_fig(fig, '03_swap_frequency_win_loss')
plt.show()

from scipy import stats
stat, pval = stats.mannwhitneyu(winner_swaps, loser_swaps, alternative='two-sided')
print(f"Mann-Whitney U test: U={stat:.0f}, p={pval:.4e}")
print(f"Winner mean swaps: {winner_swaps.mean():.2f}")
print(f"Loser mean swaps:  {loser_swaps.mean():.2f}")

In [ ]:
# Role-swap analysis: what role transitions happen most?
role_swaps = swaps.groupby(['prev_role', 'role']).size().reset_index(name='count')
role_swap_pivot = role_swaps.pivot_table(
    index='prev_role', columns='role', values='count', fill_value=0
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    role_swap_pivot,
    annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Swap Count'}
)
ax.set_title('Role Swap Matrix: From -> To')
ax.set_ylabel('Previous Role')
ax.set_xlabel('New Role')

plt.tight_layout()
save_fig(fig, '03_role_swap_matrix')
plt.show()

# Most swaps happen within the same role (counter-swapping)
same_role = swaps[swaps['prev_role'] == swaps['role']]
diff_role = swaps[swaps['prev_role'] != swaps['role']]
print(f"Same-role swaps: {len(same_role):,} ({len(same_role)/len(swaps)*100:.1f}%)")
print(f"Cross-role swaps: {len(diff_role):,} ({len(diff_role)/len(swaps)*100:.1f}%)")

## 7. Composition Matchup Analysis

How do different archetypes perform against each other? This is the classic rock-paper-scissors question in OW: does Dive beat Poke? Does Brawl beat Dive?

In [ ]:
# Build matchup table: for each match, get both teams' archetypes
match_comps = team_comps.pivot_table(
    index='MapDataId', columns='player_team',
    values='archetype', aggfunc='first'
)

# This gives us a wide table; we need to reshape for matchup analysis
# Instead, merge team_comps with itself for the same match
tc = team_comps[['MapDataId', 'player_team', 'archetype', 'team_won']].copy()

# Self-join to get opponent archetype
matchups = tc.merge(tc, on='MapDataId', suffixes=('', '_opp'))
matchups = matchups[matchups['player_team'] != matchups['player_team_opp']]

# Matchup win rates
matchup_wr = matchups.groupby(['archetype', 'archetype_opp']).agg(
    total=('team_won', 'count'),
    wins=('team_won', 'sum')
)
matchup_wr['win_rate'] = matchup_wr['wins'] / matchup_wr['total'] * 100
matchup_wr = matchup_wr.reset_index()

matchup_pivot = matchup_wr.pivot_table(
    index='archetype', columns='archetype_opp', values='win_rate'
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matchup_pivot,
    annot=True, fmt='.1f', cmap='RdYlGn', center=50,
    vmin=30, vmax=70,
    linewidths=1, linecolor=OW_COLORS['dark_blue'],
    ax=ax,
    cbar_kws={'label': 'Win Rate (%)'}
)
ax.set_title('Composition Matchup Win Rates (Row vs Column)')
ax.set_ylabel('Your Comp')
ax.set_xlabel('Opponent Comp')

plt.tight_layout()
save_fig(fig, '03_comp_matchup_matrix')
plt.show()

## 8. Summary & Coaching Implications

### Key Findings

| Topic | Finding |
|-------|---------|
| Top picked heroes | See pick rate chart above |
| Best comp archetype | See archetype win rates |
| Best hero synergies | See synergy heatmap |
| Map-hero preferences | Clear map-specific patterns exist |
| Hero swapping | See swap analysis |

### Coaching Implications

1. **Pick rate does not equal win rate**: Some of the most popular heroes may not have the highest win rates. Coaches should look at win rate, not just popularity, when building team comps.

2. **Synergy matters**: Hero pair win rates vary significantly. Building a comp around strong synergy pairs (rather than individually "meta" heroes) can provide an edge, especially at amateur levels.

3. **Map prep is real**: Hero preferences shift meaningfully across maps. Teams should have map-specific hero pools and practiced compositions rather than a one-size-fits-all approach.

4. **Swapping is a skill**: The data reveals patterns in when and how players swap. Understanding counter-swap timing (swapping to counter the enemy comp) is a coachable skill.

5. **Composition rock-paper-scissors**: The matchup matrix reveals whether the classic Dive > Poke > Brawl > Dive cycle holds in this dataset. Teams should learn at least 2 archetypes to adapt mid-match.

### For Players
- Expand your hero pool to include at least 2-3 heroes per role so you can swap effectively.
- Learn which maps favor your best heroes and request those maps in scrims.
- Coordinate hero picks with your team — synergy pairs outperform solo picks.